# 02 · Data Transformation

Turns the raw CSV into model-ready train / validation / test splits.

1. Keep only the five homework columns
2. Shuffle + split 60 / 20 / 20 exactly as in the lecture (seed 42)
3. Decide on missing-value handling, **fitting any statistic on the training split only**
4. Save the splits to `data/processed/` for `03_training_and_analysis.ipynb`

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")
OUT = DATA / "processed"
OUT.mkdir(exist_ok=True)

COLUMNS = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year",
    "fuel_efficiency_mpg",
]
TARGET = "fuel_efficiency_mpg"
FEATURES = [c for c in COLUMNS if c != TARGET]

df = pd.read_csv(DATA / "car_fuel_efficiency_2026.csv")[COLUMNS]
df.shape

(10000, 5)

## 1. Column selection & missing values

In [2]:
df.isna().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [3]:
df["horsepower"].median()

np.float64(254.0)

## 2. Train / validation / test split

In [4]:
def split_dataset(df, seed=42):
    """60/20/20 shuffled split, identical to the lecture code."""
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]
    return df_train, df_val, df_test


df_train, df_val, df_test = split_dataset(df, seed=42)
print(len(df_train), len(df_val), len(df_test))

6000 2000 2000


In [5]:
# The splits should look alike: compare the target distribution across them.
pd.DataFrame({
    "train": df_train[TARGET].describe(),
    "val": df_val[TARGET].describe(),
    "test": df_test[TARGET].describe(),
})

,train,val,test
count,6000.000000,2000.000000,2000.000000
mean,29.979767,30.016150,30.033050
std,2.934954,2.889714,2.957258
min,20.500000,19.800000,20.200000
25%,27.900000,28.000000,28.075000
50%,30.000000,30.000000,30.000000
75%,32.000000,31.900000,32.000000
max,39.800000,41.200000,39.400000


## 3. Imputation statistics (train only)

The mean used for imputation is computed on the **training** split only. Using the full dataset
would leak validation/test information into the model.

In [6]:
hp_mean_train = df_train["horsepower"].mean()
hp_mean_full = df["horsepower"].mean()
print(f"train mean: {hp_mean_train:.4f}   full-data mean (do NOT use): {hp_mean_full:.4f}")

train mean: 254.4634   full-data mean (do NOT use): 254.4460


In [7]:
def prepare_X(frame, fill_value=0.0):
    """Return the feature matrix with `horsepower` NaNs replaced by `fill_value`."""
    return frame[FEATURES].fillna(fill_value).to_numpy()


# sanity check: no NaNs left after imputation, and shapes are right
for name, part in [("train", df_train), ("val", df_val), ("test", df_test)]:
    X = prepare_X(part, hp_mean_train)
    print(name, X.shape, "NaNs left:", int(np.isnan(X).sum()))

train (6000, 4) NaNs left: 0
val (2000, 4) NaNs left: 0
test (2000, 4) NaNs left: 0


## 4. Save

In [8]:
df_train.to_csv(OUT / "train_seed42.csv", index=False)
df_val.to_csv(OUT / "val_seed42.csv", index=False)
df_test.to_csv(OUT / "test_seed42.csv", index=False)

with open(OUT / "imputation_seed42.json", "w") as f:
    json.dump({"horsepower_train_mean": hp_mean_train}, f, indent=2)

sorted(p.name for p in OUT.iterdir())

['imputation_seed42.json',
 'test_seed42.csv',
 'train_seed42.csv',
 'val_seed42.csv']

Raw NaNs are intentionally **kept** in the saved splits; the imputation strategy (0 vs. train mean) is
itself an experiment in `03_training_and_analysis.ipynb`.